# Episode Viewer
Interactive visualization of few-shot classification episodes from a retweet graph.

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
GRAPH_PATH   = "/Users/philipp/Downloads/retweet_graph.pt"
TASK         = "classification"   # "classification"  or  "neighbor_matching"
N_EPISODES   = 3      # number of episodes to visualize
N_WAY        = 2      # classes per episode
N_SHOT       = 1      # support nodes per class
N_QUERY      = 1      # query nodes per class
N_HOP        = 1      # k-hop neighborhood around episode nodes
MAX_BG_NODES = 300    # cap on background (non-episode) nodes per episode
SEED         = 42

In [22]:
import torch
import numpy as np
import plotly.graph_objects as go
from collections import defaultdict
from sklearn.decomposition import PCA
import random

rng = random.Random(SEED)
print('loading', GRAPH_PATH)
raw = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)
print('loaded')
if isinstance(raw, dict):
    x             = raw['x']
    y             = raw['y']
    edge_index    = raw['edge_index']
    label_names   = list(raw.get('label_names', []))
    feature_names = list(raw.get('feature_names', []))
else:  # torch_geometric Data object
    x             = raw.x
    y             = raw.y
    edge_index    = raw.edge_index
    label_names   = list(getattr(raw, 'label_names', []))
    feature_names = list(getattr(raw, 'feature_names', []))

N     = x.shape[0]
x_np  = x.numpy().astype(np.float32)
y_np  = y.numpy()

print(f"Nodes: {N:,}  |  Features: {x_np.shape[1]}  |  Edges: {edge_index.shape[1]:,}")
print(f"Labels ({len(label_names)}): {label_names[:12]}{'...' if len(label_names) > 12 else ''}")
print(f"Labeled nodes: {(y_np >= 0).sum():,} / {N:,}")

loading /Users/philipp/Downloads/retweet_graph.pt
loaded
Nodes: 78,672  |  Features: 390  |  Edges: 180,928
Labels (2): ['not_conservative', 'conservative']
Labeled nodes: 78,672 / 78,672


In [ ]:
# ── Build adjacency list (once) ──────────────────────────────────────────────
adj = defaultdict(set)
for s, d in zip(edge_index[0].tolist(), edge_index[1].tolist()):
    adj[s].add(d)
    adj[d].add(s)
print("Adjacency list built.")


# ── Classification episode sampler ───────────────────────────────────────────
def sample_episode(n_way, n_shot, n_query):
    by_label = defaultdict(list)
    for i, lbl in enumerate(y_np):
        if lbl >= 0:
            by_label[int(lbl)].append(i)
    eligible = [l for l, ns in by_label.items() if len(ns) >= n_shot + n_query]
    if len(eligible) < n_way:
        raise ValueError(f"Only {len(eligible)} classes have >= {n_shot+n_query} nodes; need {n_way}.")
    chosen = rng.sample(eligible, n_way)
    episode = {}
    for lbl in chosen:
        ns = by_label[lbl].copy()
        rng.shuffle(ns)
        episode[lbl] = {'support': ns[:n_shot], 'query': ns[n_shot:n_shot + n_query], 'center': None}
    return episode


# ── Neighbor-matching episode sampler ─────────────────────────────────────────
def sample_episode_nm(n_way, n_shot, n_query):
    """Each class = a center node; support/query = that center's neighbors."""
    needed = n_shot + n_query
    centers = []
    tried = set()
    all_nodes = list(range(N))
    rng.shuffle(all_nodes)
    for candidate in all_nodes:
        if candidate in tried:
            continue
        tried.add(candidate)
        neighbors = list(adj[candidate])
        if len(neighbors) >= needed:
            centers.append(candidate)
        if len(centers) == n_way:
            break
    if len(centers) < n_way:
        raise ValueError(f"Only {len(centers)} nodes have >= {needed} neighbors; need {n_way}.")
    episode = {}
    for center in centers:
        neighbors = list(adj[center])
        rng.shuffle(neighbors)
        episode[center] = {
            'support': neighbors[:n_shot],
            'query':   neighbors[n_shot:n_shot + n_query],
            'center':  center,
        }
    return episode


# ── K-hop subgraph (BFS) ─────────────────────────────────────────────────────
def k_hop_subgraph(seed_nodes, n_hops):
    seed_set = set(seed_nodes)
    visited  = set(seed_nodes)
    frontier = set(seed_nodes)
    for _ in range(n_hops):
        nxt = set()
        for node in frontier:
            for nb in adj[node]:
                if nb not in visited:
                    nxt.add(nb)
                    visited.add(nb)
                    if len(visited) - len(seed_set) >= MAX_BG_NODES:
                        break
            if len(visited) - len(seed_set) >= MAX_BG_NODES:
                break
        frontier = nxt
        if not frontier:
            break
    nodes = sorted(visited)
    n2l   = {n: i for i, n in enumerate(nodes)}
    edges = set()
    for n in nodes:
        for nb in adj[n]:
            if nb in n2l:
                a, b = min(n2l[n], n2l[nb]), max(n2l[n], n2l[nb])
                edges.add((a, b))
    return nodes, list(edges), n2l


# ── 2D layout via PCA ────────────────────────────────────────────────────────
def layout_pca(x_sub):
    emb_cols = [i for i, n in enumerate(feature_names) if n.startswith('emb_')]
    src = x_sub[:, emb_cols] if emb_cols else x_sub
    if src.shape[1] >= 2:
        return PCA(n_components=2, random_state=SEED).fit_transform(src)
    pad = np.zeros((src.shape[0], 2 - src.shape[1]))
    return np.hstack([src, pad])


# ── Hover text ───────────────────────────────────────────────────────────────
def node_hover(global_idx, role, lbl_name, x_row, max_stats=10):
    lines = [f'<b>{role}</b>', f'Node id: {global_idx}']
    if lbl_name:
        lines.append(f'Label: <b>{lbl_name}</b>')
    stats = [(i, n) for i, n in enumerate(feature_names) if not n.startswith('emb_')]
    if stats:
        lines.append('─' * 22)
        for i, name in stats[:max_stats]:
            lines.append(f'{name}: {x_row[i]:.4g}')
        if len(stats) > max_stats:
            lines.append(f'… +{len(stats) - max_stats} more features')
    elif len(x_row) > 0:
        lines.append(f'Embedding L2: {float(np.linalg.norm(x_row)):.4f}')
    return '<br>'.join(lines)


print("Helpers ready.")

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
COLORS = [
    '#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
    '#ff7f00', '#a65628', '#f781bf', '#17becf',
]

def visualize_episode(episode, ep_idx, task="classification"):
    label_ids   = sorted(episode.keys())
    color_map   = {lbl: COLORS[i % len(COLORS)] for i, lbl in enumerate(label_ids)}
    support_map = {n: lbl for lbl, sets in episode.items() for n in sets['support']}
    query_map   = {n: lbl for lbl, sets in episode.items() for n in sets['query']}
    center_map  = {sets['center']: lbl for lbl, sets in episode.items() if sets['center'] is not None}
    seed_set    = set(support_map) | set(query_map) | set(center_map)

    nodes, edges, n2l = k_hop_subgraph(sorted(seed_set), N_HOP)
    pos = layout_pca(x_np[nodes])

    fig = go.Figure()

    # edges
    ex, ey = [], []
    for a, b in edges:
        ex += [pos[a, 0], pos[b, 0], None]
        ey += [pos[a, 1], pos[b, 1], None]
    fig.add_trace(go.Scatter(
        x=ex, y=ey, mode='lines',
        line=dict(color='#cccccc', width=0.6),
        hoverinfo='none', showlegend=False,
    ))

    # background nodes
    bg = [i for i, n in enumerate(nodes) if n not in seed_set]
    if bg:
        fig.add_trace(go.Scatter(
            x=pos[bg, 0], y=pos[bg, 1], mode='markers',
            marker=dict(size=5, color='#d8d8d8', line=dict(width=0)),
            text=[node_hover(nodes[i], 'Background', '', x_np[nodes[i]]) for i in bg],
            hovertemplate='%{text}<extra></extra>',
            name='Background', legendgroup='bg',
        ))

    # per-class: support, query, and (for NM) center
    for lbl in label_ids:
        color = color_map[lbl]
        if task == "classification":
            lbl_name = label_names[lbl] if lbl < len(label_names) else str(lbl)
        else:
            lbl_name = f'Center {lbl}'

        roles = [
            ('Support', support_map, 'circle',        15),
            ('Query',   query_map,   'diamond',        15),
            ('Center',  center_map,  'star',           20),
        ]
        for role, node_map, symbol, size in roles:
            idxs = [n2l[n] for n, l in node_map.items() if l == lbl and n in n2l]
            if not idxs:
                continue
            fig.add_trace(go.Scatter(
                x=pos[idxs, 0], y=pos[idxs, 1], mode='markers',
                marker=dict(size=size, color=color, symbol=symbol,
                            line=dict(color='black', width=1.5)),
                text=[node_hover(nodes[i], f'{role} – {lbl_name}', lbl_name, x_np[nodes[i]])
                      for i in idxs],
                hovertemplate='%{text}<extra></extra>',
                name=f'{lbl_name} ({role})', legendgroup=f'cls_{lbl}',
            ))

    if task == "classification":
        ep_labels = ', '.join(
            label_names[l] if l < len(label_names) else str(l) for l in label_ids
        )
        title = f'Episode {ep_idx + 1}  —  {ep_labels}'
    else:
        title = f'Episode {ep_idx + 1}  —  NM  (centers: {", ".join(str(l) for l in label_ids)})'

    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        plot_bgcolor='white',
        hovermode='closest',
        legend=dict(itemsizing='constant', font=dict(size=11)),
        width=950, height=640,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    return fig

In [ ]:
# ── Run ──────────────────────────────────────────────────────────────────────
for ep_idx in range(N_EPISODES):
    if TASK == "neighbor_matching":
        episode = sample_episode_nm(N_WAY, N_SHOT, N_QUERY)
    else:
        episode = sample_episode(N_WAY, N_SHOT, N_QUERY)
    fig = visualize_episode(episode, ep_idx, task=TASK)
    fig.show()